# 06 - Algorithm sweep (which classifier wins on the frozen features?)

Runs ~20 classifiers on a chosen frozen feature set, 5-fold CV, seed 123, accuracy + AUC,
ranked. This is the robustness evidence that LR is a fair, representative choice and not
underselling the features.

**PCA front-end:** by default we reduce to `PCA_COMPONENTS = 50` PCs first. This (a) makes the
heavy classifiers run in seconds instead of minutes, and (b) barely changes results (the PCA
sweep showed accuracy is flat from 10 to full dimensions). Set `PCA_COMPONENTS = None` for the
full-dim run (slower, essentially the same numbers). PCA is fit INSIDE each fold, no leakage.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
DATA='../data'; OUT='../results_sensor'

FEATURE = 'frozen tiny'      # 'frozen tiny' | 'frozen base' | 'handcrafted'
PCA_COMPONENTS = 50          # None = full-dim (slower, ~same result)

paths = {'frozen tiny': f'{OUT}/frozen_tiny.npy',
         'frozen base': f'{OUT}/frozen_base.npy',
         'handcrafted': f'{DATA}/baseline_features.npy'}
X = np.load(paths[FEATURE]); y = np.load(f'{DATA}/preprocessed/labels.npy')
cv = StratifiedKFold(5, shuffle=True, random_state=123); S = 123
# PCA only makes sense when there are more features than the requested components
eff_pca = PCA_COMPONENTS if (PCA_COMPONENTS and X.shape[1] > PCA_COMPONENTS) else None
print(f'Feature set: {FEATURE} {X.shape} | PCA front-end: {eff_pca}')

Feature set: frozen tiny (148, 4096) | PCA front-end: 50


In [2]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import (LogisticRegression, RidgeClassifier, SGDClassifier,
                                  PassiveAggressiveClassifier)
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, AdaBoostClassifier,
                              BaggingClassifier, GradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier

def build(clf, needs_scale):
    steps = []
    if eff_pca: steps.append(PCA(n_components=eff_pca, random_state=S))
    if needs_scale: steps.append(StandardScaler())   # scale PCs (or raw if no PCA)
    steps.append(clf)
    return make_pipeline(*steps) if len(steps) > 1 else clf

# (name, estimator, needs_scale). Trees don't need scaling; LR-L2 ref kept unscaled.
SPEC = [
 ('LR L2 (ref)',      LogisticRegression(max_iter=2000), False),
 ('LR L1',            LogisticRegression(penalty='l1', solver='saga', max_iter=4000), True),
 ('LR ElasticNet',    LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=4000), True),
 ('Ridge',            RidgeClassifier(), True),
 ('SGD (log)',        SGDClassifier(loss='log_loss', random_state=S), True),
 ('PassiveAggr',      PassiveAggressiveClassifier(random_state=S), True),
 ('Linear SVM',       LinearSVC(max_iter=5000, random_state=S), True),
 ('RBF SVM',          SVC(kernel='rbf', random_state=S), True),
 ('Poly SVM (d3)',    SVC(kernel='poly', degree=3, random_state=S), True),
 ('Sigmoid SVM',      SVC(kernel='sigmoid', random_state=S), True),
 ('kNN k=5',          KNeighborsClassifier(n_neighbors=5), True),
 ('kNN k=15',         KNeighborsClassifier(n_neighbors=15), True),
 ('NearestCentroid',  NearestCentroid(), True),
 ('GaussianNB',       GaussianNB(), True),
 ('LDA (shrink)',     LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'), True),
 ('RandomForest',     RandomForestClassifier(n_estimators=300, random_state=S, n_jobs=-1), False),
 ('ExtraTrees',       ExtraTreesClassifier(n_estimators=300, random_state=S, n_jobs=-1), False),
 ('GradBoost',        GradientBoostingClassifier(random_state=S), False),
 ('HistGradBoost',    HistGradientBoostingClassifier(random_state=S), False),
 ('AdaBoost',         AdaBoostClassifier(random_state=S), False),
 ('Bagging',          BaggingClassifier(n_estimators=50, random_state=S, n_jobs=-1), False),
 ('MLP (64)',         MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=S), True),
 ('MLP (128,64)',     MLPClassifier(hidden_layer_sizes=(128,64), max_iter=1000, random_state=S), True),
]
print(f'{len(SPEC)} classifiers ready.')

23 classifiers ready.


In [3]:
rows = []
for name, clf, needs_scale in SPEC:
    pipe = build(clf, needs_scale)
    acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy').mean()
    try:    auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc').mean()
    except Exception: auc = float('nan')
    rows.append((name, acc, auc))
    print(f'  {name:20s} acc {acc:.3f}  AUC {auc:.3f}', flush=True)

rows.sort(key=lambda r: (-r[1], -(0 if np.isnan(r[2]) else r[2])))
print(f'\n=== {FEATURE} - ranked by accuracy (PCA={eff_pca}) ===')
for name, acc, auc in rows:
    a = f'{auc:.3f}' if not np.isnan(auc) else '  -- '
    star = '  <- LR reference' if name == 'LR L2 (ref)' else ''
    print(f'  {name:20s} acc {acc:.3f}  AUC {a}{star}')

  LR L2 (ref)          acc 0.831  AUC 0.861
  LR L1                acc 0.796  AUC 0.848
  LR ElasticNet        acc 0.783  AUC 0.845
  Ridge                acc 0.803  AUC 0.849
  SGD (log)            acc 0.749  AUC 0.830
  PassiveAggr          acc 0.750  AUC 0.820
  Linear SVM           acc 0.750  AUC 0.822
  RBF SVM              acc 0.769  AUC 0.891
  Poly SVM (d3)        acc 0.534  AUC 0.812
  Sigmoid SVM          acc 0.790  AUC 0.861
  kNN k=5              acc 0.668  AUC 0.779
  kNN k=15             acc 0.607  AUC 0.837
  NearestCentroid      acc 0.803  AUC 0.863
  GaussianNB           acc 0.716  AUC 0.830
  LDA (shrink)         acc 0.817  AUC 0.863
  RandomForest         acc 0.783  AUC 0.848
  ExtraTrees           acc 0.769  AUC 0.864
  GradBoost            acc 0.756  AUC 0.833
  HistGradBoost        acc 0.723  AUC 0.839
  AdaBoost             acc 0.736  AUC 0.837
  Bagging              acc 0.776  AUC 0.832
  MLP (64)             acc 0.749  AUC 0.815
  MLP (128,64)         acc 0.770